# 브릿지 서비스 포화도 분석

## tl;dr

- **클로드의 핵심 가설은 절반만 맞습니다.** 서비스 재고가 매우 많은 것은 사실이지만, 현재 데이터로 특정 서비스 포화가 최근 실패의 원인이라고 확정할 수는 없습니다.
- `중문설치` 독립 토큰은 14편이고 창 마감 13편 중 2편만 landed(15.4%)입니다. 다만 7월 이후 채널 전체가 함께 약해져 서비스 효과와 시기 효과가 섞였습니다.
- 과거 브릿지 4편은 모두 실패했지만, 각 발행 시기의 비교군 승률을 적용해도 4편 모두 실패할 확률이 약 52.9%입니다. 브릿지 전략을 기각할 증거가 아닙니다.
- 201~207은 '덜 붐비는 서비스'만 찾아 몰아주지 말고, 같은 고객 질문이 없는 서비스-부모 조합으로 분산해야 합니다. Q-030 개구부·생활동선 전술은 계속 차단합니다.

## Context & Methods

브릿지 글의 부모 키워드가 아니라 연결되는 문장군 서비스가 병목인지 판단하기 위한 분석입니다. 기준일은 2026-08-23 확정 통계이며 KST 날짜를 사용합니다.

### Key Assumptions

- `landed`는 저장된 verdict를 읽지 않고 D3~D14 TOP20 등장 2회 이상으로 다시 계산합니다.
- 비교 분모는 D14가 닫혔고 관측 유효일이 5일 이상인 글입니다.
- 서비스군은 서로 겹칠 수 있습니다. 예를 들어 ABS도어 문틀교체 글은 두 서비스군에 동시에 포함됩니다. 따라서 서비스군 합계는 전체 글 수가 아닙니다.
- 등록부 단어 수는 재고 밀도이며 검색 결과의 자기잠식이나 인과를 직접 증명하지 않습니다.

In [1]:
from pathlib import Path
import json, re
from datetime import date
import pandas as pd

ROOT = Path.cwd()
REGISTRY_PATH = ROOT / 'docs/strategy/POSTING_REGISTRY.json'
PERFORMANCE_PATH = ROOT / 'data/performance/post_performance.json'
TAXONOMY_PATH = ROOT / 'docs/strategy/SEO_TAXONOMY.json'
DAILY_PATH = ROOT / 'outputs/reports/daily/2026-08-24_seo_watch.md'
AS_OF = date(2026, 8, 23)

registry = json.loads(REGISTRY_PATH.read_text(encoding='utf-8'))
performance = json.loads(PERFORMANCE_PATH.read_text(encoding='utf-8'))
taxonomy = json.loads(TAXONOMY_PATH.read_text(encoding='utf-8'))
daily_text = DAILY_PATH.read_text(encoding='utf-8')

print({'registry_imported_at': registry.get('imported_at'), 'performance_updated_at': performance.get('updated_at'), 'taxonomy_updated_at': taxonomy.get('updated_at'), 'as_of': AS_OF.isoformat()})

{'registry_imported_at': '2026-07-06T05:12:46.579Z', 'performance_updated_at': '2026-08-23', 'taxonomy_updated_at': '2026-07-27', 'as_of': '2026-08-23'}


## Data

등록부의 표를 글번호 기준으로 합치고, 서비스 단어가 키워드·제목·소재·메모에서 독립 토큰으로 등장하는 글을 셉니다.

In [2]:
def registry_entries(registry_json):
    by_no = {}
    for block in registry_json.get('blocks', []):
        if block.get('type') != 'table':
            continue
        header = block.get('header', [])
        def index_of(pattern):
            return next((i for i, value in enumerate(header) if re.search(pattern, str(value))), -1)
        keyword_idx = index_of(r'타겟 키워드|추적 키워드')
        title_idx = index_of(r'제목')
        topic_idx = index_of(r'소재')
        memo_idx = index_of(r'메모')
        url_idx = index_of(r'URL')
        for row in block.get('rows', []):
            match = re.match(r'^(\d{3})', str(row[0]))
            if not match:
                continue
            post_no = match.group(1)
            current = by_no.setdefault(post_no, {'no': post_no, 'text': '', 'published': False})
            indices = [i for i in [keyword_idx, title_idx, topic_idx, memo_idx] if i >= 0]
            current['text'] += ' ' + ' '.join(str(row[i] or '') for i in indices)
            row_text = ' '.join(str(value or '') for value in row)
            if (url_idx >= 0 and 'blog.naver.com' in str(row[url_idx])) or 'blog.naver.com' in row_text:
                current['published'] = True
    for value in by_no.values():
        value['text'] = re.sub(r'\s+', ' ', value['text']).strip()
    return list(by_no.values())

entries = registry_entries(registry)
entry_by_no = {row['no']: row for row in entries}
perf_by_no = {str(row['post_no']).zfill(3): row for row in performance['posts']}

def exact_posts(term):
    pattern = re.compile(rf'(^|[\s,]){re.escape(term)}([\s,]|$)')
    return sorted(row['no'] for row in entries if pattern.search(row['text']))

def window_appearances(post):
    return sum(1 for obs in post.get('observations', []) if 3 <= obs.get('day', -1) <= 14)

def eligible(post):
    if not post or not post.get('published_at') or post.get('observed_days', 0) < 5:
        return False
    published = date.fromisoformat(post['published_at'])
    return (AS_OF - published).days >= 14

def landed(post):
    return window_appearances(post) >= 2

print({'registry_posts': len(entries), 'performance_posts': len(performance['posts']), 'eligible_closed_posts': sum(eligible(row) for row in performance['posts'])})

{'registry_posts': 199, 'performance_posts': 215, 'eligible_closed_posts': 135}


## Results

### 서비스 단어별 재고와 성과

단어별 행은 기계 검증기의 `서비스 완전 중복`과 같은 방식입니다. 글 하나가 여러 행에 중복 포함될 수 있습니다.

In [3]:
service_terms = ['중문설치', '현관중문', '3연동중문', 'ABS도어', '방문교체', '화장실문교체', '문틀교체', '문선', '몰딩', '방문턱제거']
term_rows = []
for term in service_terms:
    post_nos = exact_posts(term)
    closed = [perf_by_no[no] for no in post_nos if no in perf_by_no and eligible(perf_by_no[no])]
    recent = [post for post in closed if post['published_at'] >= '2026-07-01']
    term_rows.append({
        '서비스 단어': term,
        '등록 재고': len(post_nos),
        'URL 대기': sum(not entry_by_no[no]['published'] for no in post_nos),
        '창 마감': len(closed),
        '전체 landed': sum(landed(post) for post in closed),
        '전체 승률': round(100 * sum(landed(post) for post in closed) / len(closed), 1) if closed else None,
        '7월 이후 창 마감': len(recent),
        '7월 이후 landed': sum(landed(post) for post in recent),
        '7월 이후 승률': round(100 * sum(landed(post) for post in recent) / len(recent), 1) if recent else None,
    })
term_df = pd.DataFrame(term_rows)
term_df

,서비스 단어,등록 재고,URL 대기,창 마감,전체 landed,전체 승률,7월 이후 창 마감,7월 이후 landed,7월 이후 승률
0,중문설치,14,0,13,2,15.4,8,1,12.5
1,현관중문,20,1,12,2,16.7,6,0,0.0
2,3연동중문,37,1,23,9,39.1,9,1,11.1
3,ABS도어,29,2,17,4,23.5,8,0,0.0
4,방문교체,48,2,38,14,36.8,23,3,13.0
5,화장실문교체,16,1,9,2,22.2,5,0,0.0
6,문틀교체,21,1,16,10,62.5,5,1,20.0
7,문선,10,2,8,5,62.5,3,0,0.0
8,몰딩,3,0,2,1,50.0,1,0,0.0
9,방문턱제거,1,0,1,1,100.0,1,1,100.0


### 연결 서비스군별 비교

후보 설계에 쓰기 위해 단어를 네 개의 겹치는 서비스군으로 묶습니다. 이 표는 검색 카니발 판정표가 아니라 등록 재고와 초기 성과의 방향을 보는 표입니다.

In [4]:
service_families = {
    '중문 설치': ['중문설치', '현관중문', '3연동중문', '중문시공'],
    '방문·ABS 교체': ['방문교체', 'ABS도어', '화장실문교체', '문짝교체', '도어교체'],
    '문틀·문선 마감': ['문틀교체', '문틀', '문선', '몰딩', '걸레받이', '천장몰딩'],
    '문턱·바닥 경계': ['방문턱제거', '문턱', '바닥마감', '레일'],
}
family_rows = []
for family, terms in service_families.items():
    post_nos = sorted(set().union(*(set(exact_posts(term)) for term in terms)))
    closed = [perf_by_no[no] for no in post_nos if no in perf_by_no and eligible(perf_by_no[no])]
    july = [post for post in closed if post['published_at'] >= '2026-07-01']
    w5_plus = [post for post in closed if post['published_at'] >= '2026-07-19']
    family_rows.append({
        '서비스군': family,
        '등록 재고': len(post_nos),
        'URL 대기': sum(not entry_by_no[no]['published'] for no in post_nos),
        '전체 창 마감': len(closed),
        '전체 landed': sum(landed(post) for post in closed),
        '전체 승률': round(100 * sum(landed(post) for post in closed) / len(closed), 1) if closed else None,
        '7월 이후 모수': len(july),
        '7월 이후 landed': sum(landed(post) for post in july),
        '7월 이후 승률': round(100 * sum(landed(post) for post in july) / len(july), 1) if july else None,
        '7/19 이후 모수': len(w5_plus),
        '7/19 이후 landed': sum(landed(post) for post in w5_plus),
    })
family_df = pd.DataFrame(family_rows)
family_df

,서비스군,등록 재고,URL 대기,전체 창 마감,전체 landed,전체 승률,7월 이후 모수,7월 이후 landed,7월 이후 승률,7/19 이후 모수,7/19 이후 landed
0,중문 설치,62,3,41,11,26.8,18,2,11.1,4,1
1,방문·ABS 교체,64,3,47,17,36.2,27,4,14.8,13,1
2,문틀·문선 마감,47,3,39,19,48.7,21,5,23.8,9,1
3,문턱·바닥 경계,11,0,10,3,30.0,6,2,33.3,3,1


### 과거 브릿지 4편과 같은 시기 비교군

각 브릿지 글 발행일 앞뒤 3일의 창 마감 글을 비교군으로 둡니다. 같은 글이 인접 비교군에 중복될 수 있으므로 네 행을 합산하지 않습니다.

In [5]:
bridge_posts = ['116', '136', '153', '165']
eligible_numeric = [post for post in performance['posts'] if str(post.get('post_no', '')).isdigit() and eligible(post)]
bridge_rows = []
zero_success_probability = 1.0
for post_no in bridge_posts:
    bridge = perf_by_no[post_no]
    bridge_date = date.fromisoformat(bridge['published_at'])
    peers = [post for post in eligible_numeric if abs((date.fromisoformat(post['published_at']) - bridge_date).days) <= 3]
    peer_rate = sum(landed(post) for post in peers) / len(peers)
    zero_success_probability *= (1 - peer_rate)
    bridge_rows.append({
        '글번호': post_no,
        '발행일': bridge['published_at'],
        'D3~D14 등장': window_appearances(bridge),
        '비교군 모수': len(peers),
        '비교군 landed': sum(landed(post) for post in peers),
        '비교군 승률': round(peer_rate * 100, 1),
    })
bridge_df = pd.DataFrame(bridge_rows)
display(bridge_df)
print({'같은 시기 승률을 가정할 때 4편 모두 실패할 확률': round(zero_success_probability * 100, 1)})

,글번호,발행일,D3~D14 등장,비교군 모수,비교군 landed,비교군 승률
0,116,2026-06-30,1,22,5,22.7
1,136,2026-07-07,1,19,5,26.3
2,153,2026-07-13,0,14,1,7.1
3,165,2026-07-23,0,7,0,0.0


{'같은 시기 승률을 가정할 때 4편 모두 실패할 확률': 52.9}


### 최근 분류된 세부 전술

SEO 택소노미가 있는 글만 세부 전술별로 봅니다. 오래된 글은 분류가 비어 있어 전체 서비스 역사로 확대 해석하지 않습니다.

In [6]:
cluster_rows = []
for cluster in taxonomy['clusters']:
    post_nos = [
        key.split(':', 1)[1].zfill(3)
        for key, assignment in taxonomy['assignments'].items()
        if key.startswith('post:') and cluster['cluster_id'] in assignment.get('cluster_ids', [])
    ]
    closed = [perf_by_no[no] for no in post_nos if no in perf_by_no and eligible(perf_by_no[no])]
    cluster_rows.append({
        '클러스터': cluster['label'],
        '분류 글': len(post_nos),
        '창 마감': len(closed),
        'landed': sum(landed(post) for post in closed),
        'faded': sum(not landed(post) for post in closed),
    })
cluster_df = pd.DataFrame(cluster_rows).sort_values(['창 마감', '분류 글'], ascending=False)
cluster_df

,클러스터,분류 글,창 마감,landed,faded
3,방문·ABS도어 교체 범위,11,7,1,6
11,두짝미서기와 개구부 조건,10,7,1,6
7,손잡이·경첩과 문짝 교체 신호,4,4,1,3
4,화장실·세탁실 문 수분 손상,4,3,0,3
9,방문 시트지와 바탕 상태,4,2,0,2
5,문선·몰딩 마감과 색상,3,2,0,2
10,반려동물 현관 중문 생활 동선,2,1,0,1
0,아파트 중문 설치 비용 판단,1,1,0,1
1,신축아파트 옵션·가벽 조건,1,1,0,1
6,실내문 미닫이문 생활 조건,1,1,0,1


## Takeaways

1. **서비스 재고 병목은 실제입니다.** 겹치는 서비스군 기준 등록 재고는 중문 설치 62편, 방문·ABS 교체 64편, 문틀·문선 마감 47편입니다. 부모 키워드가 새로워도 도착 질문이 같으면 신규 글이 아닙니다.
2. **하지만 서비스 포화가 성과 하락의 원인이라는 주장은 미확정입니다.** 7월 이후 전체 적격 글 승률이 16.4%(11/67)로 내려갔고, 7월 19일 이후에는 7.7%(2/26)입니다. 서비스군별 최근 표본은 작고 서로 겹칩니다.
3. **문틀·문선 마감은 상대적으로 유망한 탐색축입니다.** 전체 창 마감 39편 중 19편 landed(48.7%), 7월 이후 21편 중 5편(23.8%)으로 다른 대형 서비스군보다 높았습니다. 다만 7월 19일 이후는 9편 중 1편이라 확정적 우위는 아닙니다.
4. **과거 브릿지 4편은 실패했지만 전략 폐기 근거가 아닙니다.** 116·136은 각각 1회 등장했고, 153·165가 발행된 시기의 비교군 자체가 7.1%와 0%였습니다.
5. **다음 결정 규칙:** 부모 수요 + 실제 의존관계 + 같은 고객 질문 없음 + 서비스군 분산을 동시에 적용합니다. 단순 등록 편수가 적다는 이유만으로 소재를 승격하지 않습니다.